# 06. Baseline Rule-Based Model
**From Space to Action** — Agricultural Drought Early Warning
Reference operational model using VCI thresholds.

In [ ]:
# === Google Colab Setup & Environment Detection ===
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

# Detect environment (Google Colab vs Local)
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print(" Running in Google Colab environment")
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        # Check if project folder exists in Drive
        candidates = [
            '/content/drive/MyDrive/venus',
            '/content/drive/MyDrive/FromSpaceToAction',
            '/content/venus',
            '/content/FromSpaceToAction',
            '.'
        ]
        PROJECT_DIR = '.'
        for c in candidates:
            if os.path.exists(c) and os.path.exists(os.path.join(c, 'src')):
                PROJECT_DIR = c
                break
        print(f"📁 Project root set to: {PROJECT_DIR}")
    except Exception as e:
        print("Note: Drive mount skipped or failed, using local Colab directory.")
        PROJECT_DIR = '.'
else:
    print("💻 Running in local environment")
    PROJECT_DIR = '.'

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')
REPORTS_DIR = os.path.join(OUTPUTS_DIR, 'reports')

for d in [DATA_DIR, os.path.join(DATA_DIR, 'targets'), os.path.join(DATA_DIR, 'features'),
          MODELS_DIR, OUTPUTS_DIR, REPORTS_DIR, os.path.join(OUTPUTS_DIR, 'maps')]:
    os.makedirs(d, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(" Setup verified.")

In [ ]:
# === Load Dataset & Automatic Fallback ===
import pandas as pd
import numpy as np

train_path = os.path.join(DATA_DIR, 'targets', 'train.csv')
val_path = os.path.join(DATA_DIR, 'targets', 'val.csv')
test_path = os.path.join(DATA_DIR, 'targets', 'test.csv')

if not os.path.exists(train_path):
    print("⚠️ Precomputed dataset not found. Generating pilot dataset for Oromia...")
    prep_script = os.path.join(PROJECT_DIR, 'scripts', 'prepare_dataset.py')
    if os.path.exists(prep_script):
        !python {prep_script}
    else:
        # Inline fallback dataset generation
        print("Running inline data generator...")
        import math, random
        from datetime import datetime, timedelta
        
        # Inline generation of 500 cells x 72 timesteps for immediate execution
        random.seed(42)
        records = []
        dates = [datetime(2018, 1, 1) + timedelta(days=10*i) for i in range(72)]
        for cid in range(500):
            lat, lon = 7.5 + random.random()*2.0, 38.5 + random.random()*2.0
            elev = 1500.0 + random.random()*1000.0
            for i, d in enumerate(dates):
                m = d.month
                season = math.sin(2.0 * math.pi * m / 12.0)
                ndvi = max(0.1, min(0.9, 0.4 + 0.2*season + (random.random()-0.5)*0.1))
                vci = max(5.0, min(95.0, 55.0 + 25.0*season + (random.random()-0.5)*15.0))
                rain = max(0.0, 50.0 + 40.0*season + (random.random()-0.5)*20.0)
                sm = max(0.05, min(0.45, 0.25 + 0.1*season + (random.random()-0.5)*0.05))
                c_class = 3 if vci <= 20 else (2 if vci <= 35 else (1 if vci <= 40 else 0))
                records.append({
                    'cell_id': cid, 'date': d.strftime('%Y-%m-%d'), 'lat': lat, 'lon': lon, 'elevation': elev,
                    'month_sin': math.sin(2*math.pi*m/12), 'month_cos': math.cos(2*math.pi*m/12),
                    'ndvi': ndvi, 'evi': ndvi*0.8, 'ndmi': ndvi*0.7, 'vci': vci,
                    'rain_30d': rain, 'rain_60d': rain*1.8, 'rain_anomaly': (rain-50)/25,
                    'smap_sm': sm, 'smap_anomaly': (sm-0.25)/0.1, 'temp_anomaly': 0.0, 'lst': 298.0,
                    'ndvi_lag_1': ndvi, 'ndvi_lag_2': ndvi, 'ndvi_rollmean_3': ndvi,
                    'vci_lag_1': vci, 'vci_lag_2': vci, 'vci_rollmean_3': vci, 'vci_trend_4': 0.0,
                    'target': c_class, 'split': 'Train' if d.strftime('%Y-%m-%d') <= '2022-01-01' else ('Val' if d.strftime('%Y-%m-%d') <= '2023-07-01' else 'Test')
                })
        df_all = pd.DataFrame(records)
        df_all[df_all['split'] == 'Train'].to_csv(train_path, index=False)
        df_all[df_all['split'] == 'Val'].to_csv(val_path, index=False)
        df_all[df_all['split'] == 'Test'].to_csv(test_path, index=False)
        print("✅ Fallback dataset generated.")

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(f"📊 Dataset Loaded Successfully:")
print(f"  - Train observations: {len(train_df):,}")
print(f"  - Validation observations: {len(val_df):,}")
print(f"  - Held-out Test observations: {len(test_df):,}")

# Feature columns
FEATURE_COLS = [c for c in train_df.columns if c not in ['cell_id', 'date', 'current_class', 'target_lead_1', 'target_lead_2', 'target_lead_3', 'target', 'split']]
TARGET_COL = 'target'

print(f"\n🎯 Features ({len(FEATURE_COLS)}):", FEATURE_COLS)
print("📈 Class distribution in training set:")
print(train_df[TARGET_COL].value_counts(normalize=True).rename({0: 'Normal', 1: 'Watch', 2: 'Warning', 3: 'Severe'}))

In [ ]:
# Run VCI thresholding baseline
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def predict_vci(vci):
    preds = np.zeros(len(vci), dtype=int)
    preds[(vci <= 40) & (vci > 35)] = 1
    preds[(vci <= 35) & (vci > 20)] = 2
    preds[vci <= 20] = 3
    return preds

test_preds = predict_vci(test_df['vci'].values)
print("Classification Report on Held-Out Test Set:")
print(classification_report(test_df[TARGET_COL], test_preds, target_names=['Normal', 'Watch', 'Warning', 'Severe']))
print(f"Macro F1 Score: {f1_score(test_df[TARGET_COL], test_preds, average='macro'):.4f}")